# Comprehensive Exploratory Data Analysis (EDA) & Multi-Station Visualization
## 10-Year High-Resolution Hanoi Meteorological Time Series (`6,312,384` Records)

- **Course:** Advanced AI & Data Science (Chuyên đề TTNT & KHDL — Lớp 65TTNT, Trường Đại học Thủy Lợi)
- **Dataset:** Hanoi Weather Multi-Station Benchmark (`hanguynthun/hanoi-weather-multi-station-benchmark`)
- **Scale:** 6 Monitoring Stations (`st01`–`st06`) × 1,052,064 5-minute timestamps (2016–2025) = **6,312,384 station-observations**
- **Architecture Target:** TimeXer / DINO-v3 Multi-Station Spatio-Temporal Temperature Forecasting (60-minute horizon)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.style.use('ggplot')

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Locate dataset dynamically on Kaggle or local environment
kaggle_csvs = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
preferred = [p for p in kaggle_csvs if 'weather_wide_synced' in os.path.basename(p)]
candidates = preferred + kaggle_csvs + [
    "weather_wide_synced.csv",
    "data/processed/weather_wide_synced.csv",
    "data/processed/weather_wide_synced_sample.csv",
    "../data/processed/weather_wide_synced_sample.csv",
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("Dataset CSV file not found in /kaggle/input or local workspace!")

print(f"[OK] Loading dataset from: {csv_path}")
sample_cols = pd.read_csv(csv_path, nrows=5).columns
dtype_map = {c: 'float32' for c in sample_cols if c != 'timestamp'}

df_wide = pd.read_csv(csv_path, dtype=dtype_map)
df_wide['timestamp'] = pd.to_datetime(df_wide['timestamp'])
df_wide = df_wide.sort_values('timestamp').reset_index(drop=True)

stations = sorted({c.split('_')[0] for c in df_wide.columns if c.endswith('_temperature')})
total_obs = len(df_wide) * max(len(stations), 1)
print(f"[OK] Loaded {len(df_wide):,} wide 5-min rows x {len(stations)} stations = {total_obs:,} records ({df_wide.shape[1]} columns)")
print(f"[OK] Detected stations: {stations}")
print(f"[OK] Time span: {df_wide['timestamp'].min()} --> {df_wide['timestamp'].max()}")

# Hourly resampled dataframe for crisp macro trend, STL, and lead-lag analysis
df_wide_ts = df_wide.set_index('timestamp')
df_hourly = df_wide_ts.resample('1h').mean(numeric_only=True)

core_vars = [
    'temperature', 'humidity', 'dew_point', 'apparent_temperature',
    'surface_pressure', 'precipitation', 'cloud_cover',
    'wind_speed_10m', 'wind_gusts_10m', 'solar_radiation', 'direct_radiation'
]
ref_station = stations[0] if stations else ''
df = pd.DataFrame({'timestamp': df_hourly.index})
for v in core_vars:
    col_name = f"{ref_station}_{v}" if ref_station else v
    if col_name in df_hourly.columns:
        df[v] = df_hourly[col_name].values

print(f"[OK] Hourly view ready ({len(df):,} hours) using reference station: {ref_station or 'default'}")
df_wide.head()

### 1. Data Schema, Completeness & Multi-Station Summary

In [ ]:
total_nans = int(df_wide.isnull().sum().sum())
dup_timestamps = int(df_wide['timestamp'].duplicated().sum())
print(f"Total missing (NaN) cells across all {df_wide.shape[1]} columns: {total_nans}")
print(f"Duplicate timestamps: {dup_timestamps}")

temp_cols = [f"{s}_temperature" for s in stations if f"{s}_temperature" in df_wide.columns]
print("\nTemperature Summary Across All 6 Monitoring Stations (°C):")
display(df_wide[temp_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

print(f"\nCovariate Summary for Reference Station ({ref_station}):")
df.drop(columns=['timestamp']).describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

### 2. Multi-Station Spatial Consistency & Microclimate Dynamics (`st01`–`st06`)

In [ ]:
freq = 'W' if len(df_hourly) > 500 else 'D'
freq_label = 'Weekly' if freq == 'W' else 'Daily'
station_agg = df_wide_ts[temp_cols].resample(freq).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=130, gridspec_kw={'width_ratios': [1.8, 1]})

colors = ['#1d4ed8', '#059669', '#d97706', '#dc2626', '#7c3aed', '#0891b2']
for i, col in enumerate(temp_cols):
    ax1.plot(station_agg.index, station_agg[col], lw=1.4, alpha=0.88, color=colors[i % len(colors)], label=col.replace('_temperature', '').upper())

ax1.set_title(f'6-Station {freq_label} Mean Temperature Trajectories (Hanoi)', fontweight='bold')
ax1.set_ylabel('Temperature (°C)', fontweight='bold')
ax1.set_xlabel('Timestamp', fontweight='bold')
ax1.legend(ncol=3, loc='lower right', frameon=True)
ax1.grid(True, linestyle='--', alpha=0.5)

station_corr = df_wide[temp_cols].corr()
labels = [c.replace('_temperature', '').upper() for c in station_corr.index]
im = ax2.imshow(station_corr.values, cmap='YlGnBu', vmin=0.85, vmax=1.0)
ax2.set_xticks(range(len(labels)))
ax2.set_yticks(range(len(labels)))
ax2.set_xticklabels(labels, fontweight='bold')
ax2.set_yticklabels(labels, fontweight='bold')
for r in range(len(labels)):
    for c in range(len(labels)):
        val = station_corr.values[r, c]
        txt_color = 'white' if val > 0.94 else '#0f172a'
        ax2.text(c, r, f"{val:.3f}", ha='center', va='center', color=txt_color, fontsize=9, fontweight='bold')
fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
ax2.set_title('Inter-Station Temperature Spatial Correlation', fontweight='bold')

plt.tight_layout()
plt.show()

### 3. Macro Climate Trajectory & Exogenous Drivers

In [ ]:
df_ts = df.set_index('timestamp')
agg_freq = 'W' if len(df_ts) > 500 else '6h'
weekly = df_ts.resample(agg_freq).agg({
    'temperature': ['mean', 'min', 'max'],
    'humidity': 'mean',
    'solar_radiation': 'mean',
    'surface_pressure': 'mean'
}).dropna()
dates = weekly.index

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 10), sharex=True, dpi=130)

ax1.fill_between(dates, weekly['temperature']['min'], weekly['temperature']['max'], color='#93c5fd', alpha=0.45, label='Min-Max Envelope')
ax1.plot(dates, weekly['temperature']['mean'], color='#1e3a8a', lw=1.6, label='Mean Temperature (°C)')
ax1.set_ylabel('Temperature (°C)', fontweight='bold')
ax1.set_title(f'Macro Climate Trajectory & Extreme Envelopes ({ref_station.upper()})', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right')

ax2_twin = ax2.twinx()
l1 = ax2.plot(dates, weekly['solar_radiation']['mean'], color='#d97706', lw=1.3, label='Solar Radiation (W/m²)')
l2 = ax2_twin.plot(dates, weekly['humidity']['mean'], color='#059669', lw=1.3, linestyle='--', label='Relative Humidity (%)')
ax2.set_ylabel('Solar Rad (W/m²)', color='#d97706', fontweight='bold')
ax2_twin.set_ylabel('Humidity (%)', color='#059669', fontweight='bold')
lines = l1 + l2
ax2.legend(lines, [l.get_label() for l in lines], loc='upper right')

ax3.plot(dates, weekly['surface_pressure']['mean'], color='#475569', lw=1.3, label='Surface Pressure (hPa)')
ax3.set_ylabel('Pressure (hPa)', fontweight='bold')
ax3.set_xlabel('Timestamp', fontweight='bold')
ax3.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 4. 24-Hour Diurnal Cycle & Thermal Phase Lag
Demonstrating the physical phase delay between peak solar irradiance (~12:00 PM) and peak air temperature (~14:00–15:00 PM).

In [ ]:
df['hour'] = df['timestamp'].dt.hour
diurnal = df.groupby('hour').agg({
    'temperature': 'mean',
    'solar_radiation': 'mean',
    'humidity': 'mean'
})

fig, ax_t = plt.subplots(figsize=(12, 5), dpi=130)
ax_rad = ax_t.twinx()

hours = diurnal.index.values
p1 = ax_t.plot(hours, diurnal['temperature'], color='#b91c1c', lw=2.5, marker='o', label='Temperature (°C)')
p2 = ax_rad.plot(hours, diurnal['solar_radiation'], color='#d97706', lw=2.0, linestyle='--', marker='s', label='Solar Radiation (W/m²)')

ax_t.set_xlabel('Hour of Day (0–23)', fontweight='bold')
ax_t.set_ylabel('Temperature (°C)', color='#b91c1c', fontweight='bold')
ax_rad.set_ylabel('Solar Radiation (W/m²)', color='#d97706', fontweight='bold')
ax_t.set_xticks(range(24))
ax_t.set_title('24-Hour Diurnal Profile Demonstrating Thermal-Radiative Lag', fontsize=12, fontweight='bold')
ax_t.grid(True, linestyle='--', alpha=0.5)

lines = p1 + p2
ax_t.legend(lines, [l.get_label() for l in lines], loc='upper left')
plt.tight_layout()
plt.show()

### 5. Monthly Seasonality & Temperature Distribution

In [ ]:
df['month'] = df['timestamp'].dt.month
months_present = sorted(df['month'].dropna().unique())
box_data = [df.loc[df['month'] == m, 'temperature'].dropna().values for m in months_present]

fig, ax = plt.subplots(figsize=(12, 5), dpi=130)
bp = ax.boxplot(box_data, tick_labels=[f"M{int(m)}" for m in months_present], patch_artist=True)
cmap = plt.cm.coolwarm
for idx, patch in enumerate(bp['boxes']):
    ratio = idx / max(len(bp['boxes']) - 1, 1)
    # peak summer in middle months (May-Sep)
    heat = np.sin(ratio * np.pi)
    patch.set_facecolor(cmap(heat))
    patch.set_alpha(0.75)

ax.set_title('Monthly Temperature Distributions (Hanoi Benchmark)', fontsize=12, fontweight='bold')
ax.set_xlabel('Month', fontweight='bold')
ax.set_ylabel('Hourly Temperature (°C)', fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 6. Cross-Correlation Matrix: Endogenous Target vs Exogenous Covariates

In [ ]:
num_cols = [
    c for c in [
        'temperature', 'humidity', 'dew_point', 'apparent_temperature',
        'surface_pressure', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'solar_radiation'
    ] if c in df.columns
]
corr = df[num_cols].corr()
vals = corr.values.copy()
mask = np.triu(np.ones_like(vals, dtype=bool), k=1)
masked_vals = np.ma.masked_where(mask, vals)

fig, ax = plt.subplots(figsize=(10, 7.5), dpi=130)
im = ax.imshow(masked_vals, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(num_cols, fontsize=9)
for r in range(len(num_cols)):
    for c in range(len(num_cols)):
        if not mask[r, c]:
            v = vals[r, c]
            txt_color = 'white' if abs(v) > 0.6 else '#0f172a'
            ax.text(c, r, f"{v:.2f}", ha='center', va='center', color=txt_color, fontsize=8.5, fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Pearson Correlation Matrix: Target vs Exogenous Covariates ({ref_station.upper()})', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### 7. Time Series Decomposition (Trend, Diurnal Seasonality & Residuals)

In [ ]:
window_hours = min(len(df_ts), 24 * 60)
sample_ts = df_ts['temperature'].iloc[:window_hours]
rolling_trend = sample_ts.rolling(window=24, min_periods=1, center=True).mean()
detrended = sample_ts - rolling_trend
hourly_seasonal = detrended.groupby(detrended.index.hour).transform('mean')
residual = detrended - hourly_seasonal

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True, dpi=130)
axes[0].plot(sample_ts, color='#1e3a8a', lw=1.2)
axes[0].set_ylabel('Observed', fontweight='bold')
axes[0].set_title(f'Hourly Temperature Decomposition ({max(1, window_hours // 24)}-Day Window)', fontsize=12, fontweight='bold')

axes[1].plot(rolling_trend, color='#d97706', lw=1.5)
axes[1].set_ylabel('Trend (24h MA)', fontweight='bold')

axes[2].plot(hourly_seasonal, color='#059669', lw=1.0)
axes[2].set_ylabel('Seasonal (24h)', fontweight='bold')

axes[3].plot(residual, color='#64748b', lw=0.8)
axes[3].set_ylabel('Residual', fontweight='bold')
axes[3].set_xlabel('Timestamp', fontweight='bold')

plt.tight_layout()
plt.show()

### 8. Lead-Lag Cross-Correlation Analysis
Quantifying how many hours exogenous drivers lead target air temperature.

In [ ]:
max_lag = min(24, max(3, len(df) // 4))
lags = np.arange(-max_lag, max_lag + 1)
ccf_rad = [df['temperature'].corr(df['solar_radiation'].shift(-lag)) for lag in lags]
ccf_hum = [df['temperature'].corr(df['humidity'].shift(-lag)) for lag in lags]

plt.figure(figsize=(10, 4.5), dpi=130)
plt.plot(lags, ccf_rad, color='#d97706', lw=2, marker='o', markersize=3, label='Temp vs Solar Radiation')
plt.plot(lags, ccf_hum, color='#059669', lw=2, linestyle='--', marker='s', markersize=3, label='Temp vs Relative Humidity')
plt.axvline(0, color='gray', linestyle=':')
plt.xlabel('Lag (Hours, Negative = Exogenous Leads Target)', fontweight='bold')
plt.ylabel('Cross-Correlation', fontweight='bold')
plt.title(f'Lead-Lag Cross-Correlation Analysis (±{max_lag}-Hour Horizon)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 9. Key Findings & Modeling Readiness for TimeXer / DINO-v3
1. **Massive Multi-Station Scale:** **6,312,384 observations** across 6 synchronized stations (`st01`–`st06`) at 5-minute resolution over 10 continuous years (2016–2025), enabling deep vision-backbone models (**DINO-v3**) to capture complex spatio-temporal patterns without overfitting.
2. **Strong Spatial Synergy:** High inter-station correlation across the 6 monitoring locations validates representing the 72-step × 6-station × 11-feature window as a 3-channel spatial-temporal tensor (`H=66, W=72`).
3. **Physical Lead-Lag Causality:** Peak solar radiation consistently leads peak air temperature by 2–3 hours, providing strong predictive signal for multi-variate cross-attention over 60-minute horizons.
4. **Data Integrity:** Zero missing values (`0% NaN`), strictly monotonic 5-minute intervals, ready for reproducible train/val/test chronological splitting.